In [19]:
import dspy
import boto3
region_name = "us-east-1"

lm = dspy.LM(
    model='bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0',
    max_tokens=4096
)
dspy.settings.configure(lm=lm)


### Bedrock Optimization

In [ ]:
import dspy
from typing import Dict, List, Optional
from pydantic import BaseModel, Field
from enum import Enum
from typing import Annotated, Literal

class PersonaTone(str, Enum):
    FORMAL = "formal"
    CASUAL = "casual"
    TECHNICAL = "technical"
    EMPATHETIC = "empathetic"
    ANALYTICAL = "analytical"
    CREATIVE = "creative"
    AUTHORITATIVE = "authoritative"
    EDUCATIONAL = "educational"


class RolePersona(BaseModel):
    """Defines a specific role persona for the LLM"""
    name: str = Field(..., description="Role name/title")
    background: str = Field(..., description="Professional background and expertise")
    personality_traits: List[str] = Field(..., description="Key personality characteristics")
    communication_style: PersonaTone = Field(..., description="Primary communication tone")
    domain_expertise: List[str] = Field(..., description="Areas of expertise")
    values: List[str] = Field(..., description="Core values that guide decisions")
    typical_phrases: List[str] = Field(..., description="Characteristic phrases or expressions")
    knowledge_boundaries: List[str] = Field(..., description="What this persona doesn't know")


class RolePromptSignature(dspy.Signature):
    """Execute task while maintaining a specific role persona"""
    
    role_description = dspy.InputField(desc="Detailed description of the role to adopt")
    task = dspy.InputField(desc="The task to complete in character")
    context = dspy.InputField(desc="Additional context for the task")
    output = dspy.OutputField(desc="Response maintaining the specified persona")


class PersonaAgent(dspy.Module):
    """Agent that adopts specific personas for different tasks"""
    
    def __init__(self, persona: RolePersona):
        super().__init__()
        self.persona = persona
        self.role_executor = dspy.ChainOfThought(RolePromptSignature)
        
    def _build_role_description(self) -> str:
        """Build a comprehensive role description from persona"""
        return f"""
        You are {self.persona.name}, with the following characteristics:

        BACKGROUND:
        {self.persona.background}

        PERSONALITY TRAITS:
        {', '.join(self.persona.personality_traits)}

        COMMUNICATION STYLE:
        You communicate in a {self.persona.communication_style.value} manner.

        AREAS OF EXPERTISE:
        {', '.join(self.persona.domain_expertise)}

        CORE VALUES:
        {', '.join(self.persona.values)}

        CHARACTERISTIC PHRASES:
        - {chr(10).join(f'"{phrase}"' for phrase in self.persona.typical_phrases)}

        KNOWLEDGE BOUNDARIES:
        You acknowledge when topics fall outside your expertise, specifically:
        {', '.join(self.persona.knowledge_boundaries)}

        Maintain this persona consistently throughout your response.
        """
        
    def forward(self, task: str, context: str = "") -> str:
        """Execute task while maintaining persona"""
        role_description = self._build_role_description()
        result = self.role_executor(
            role_description=role_description,
            task=task,
            context=context
        )
        return result.output


class ClassificationOutput(BaseModel):
    """Output of the classification task"""

    category: Annotated[Literal['OUT_OF_SCOPE', "SPAM", "POLICY_INQUIRY"], Field(..., description="The category of the classification")]
    confidence_score: Annotated[float, Field(..., description="The confidence score for the classification")]
    rationale: Annotated[str, Field(..., description="The rationale for the classification in less than 20 words")]

class RolePromptCustomSignature(dspy.Signature):
    """Execute task while maintaining a specific role persona"""
    
    role_persona: RolePersona = dspy.InputField(desc="Detailed description of the role to adopt")
    task: str = dspy.InputField(desc="The task to complete in character")
    context: str = dspy.InputField(desc="Additional context for the task")
    classification_output: ClassificationOutput = dspy.OutputField(desc="Response maintaining the specified persona")

class PersonaPromptSignature(dspy.Signature):
    """Predict the Persona for a given task"""
    task = dspy.InputField(desc="The task to complete in character")
    persona: RolePersona = dspy.OutputField(desc="The persona to adopt for the task")


class PersonaMetaOptimizer(dspy.Module):
    """Find an Optimum Persona Prompt Given a Particlar task, inputs and outputs"""

    def __init__(self, task: str):
        self.task = task
        self.persona_predictor = dspy.ChainOfThought(PersonaPromptSignature)
        self.persona = None
        self.persona_agent = dspy.ChainOfThought(RolePromptCustomSignature)

    def forward(self, context: str = "") -> str:
        """Predict the Persona for a given task"""
        self.persona = self.persona_predictor(task=self.task)
        return self.persona_agent(role_persona=self.persona, task=self.task, context=context)


### Thought Sequence 


task -->  RolePersona 

In [58]:
model = PersonaMetaOptimizer(task="You have to judge the classification of the email given its body and subject")

In [59]:
pred  = model(context="""	
Your message to kamalseetul3@gmail.com couldn't be delivered.
kamalseetul3 wasn't found at gmail.com.
SBNHotelJoiners 	Office 365 	kamalseetul3 	
Action Required 		Recipient 	
Unknown To address 		
How to Fix It	 
The address may be misspelled or may not exist. Try one or more of the following:	 
*	Send the message again following these steps: In Outlook, open this non-delivery report (NDR) and choose Send Again from the Report ribbon. In Outlook on the web, select this NDR, then select the link "To send this message again, click here." Then delete and retype the entire recipient address. If prompted with an Auto-Complete List suggestion don't select it. After typing the complete address, click Send.
*	Contact the recipient (by phone, for example) to check that the address exists and is correct.
*	The recipient may have set up email forwarding to an incorrect address. Ask them to check that any forwarding they've set up is working correctly.
*	Clear the recipient Auto-Complete List in Outlook or Outlook on the web by following the steps in this article: Fix email delivery issues for error code 5.1.1 in Office 365  , and then send the message again. Retype the entire recipient address before selecting Send.
If the problem continues, forward this message to your email admin. If you're an email admin, refer to the More Info for Email Admins section below.	 
Was this helpful? Send feedback to Microsoft  . """)

In [61]:
email_context = """Your message to kamalseetul3@gmail.com couldn't be delivered.
kamalseetul3 wasn't found at gmail.com.
SBNHotelJoiners 	Office 365 	kamalseetul3 	
Action Required 		Recipient 	
Unknown To address 		
How to Fix It	 
The address may be misspelled or may not exist. Try one or more of the following:	 
*	Send the message again following these steps: In Outlook, open this non-delivery report (NDR) and choose Send Again from the Report ribbon. In Outlook on the web, select this NDR, then select the link "To send this message again, click here." Then delete and retype the entire recipient address. If prompted with an Auto-Complete List suggestion don't select it. After typing the complete address, click Send.
*	Contact the recipient (by phone, for example) to check that the address exists and is correct.
*	The recipient may have set up email forwarding to an incorrect address. Ask them to check that any forwarding they've set up is working correctly.
*	Clear the recipient Auto-Complete List in Outlook or Outlook on the web by following the steps in this article: Fix email delivery issues for error code 5.1.1 in Office 365  , and then send the message again. Retype the entire recipient address before selecting Send.
If the problem continues, forward this message to your email admin. If you're an email admin, refer to the More Info for Email Admins section below.	 
Was this helpful? Send feedback to Microsoft  ."""

In [60]:
pred.classification_output.model_dump()

{'category': 'POLICY_INQUIRY',
 'confidence_score': 0.95,
 'rationale': 'System-generated delivery failure notification with legitimate troubleshooting steps for email policy'}

In [62]:
model.persona.persona.model_dump()

{'name': 'Email Classification Specialist',
 'background': 'Information Management Professional with 10+ years experience in email systems and content classification',
 'personality_traits': ['detail-oriented',
  'systematic',
  'methodical',
  'precise',
  'objective'],
 'communication_style': <PersonaTone.ANALYTICAL: 'analytical'>,
 'domain_expertise': ['email classification systems',
  'content analysis',
  'information architecture',
  'business communication',
  'data categorization'],
 'values': ['accuracy',
  'efficiency',
  'consistency',
  'systematic approach',
  'clear organization'],
 'typical_phrases': ['Based on the content analysis...',
  'The key indicators suggest...',
  'Following our classification framework...',
  'The primary category appears to be...',
  'Taking into account the subject line and body...'],
 'knowledge_boundaries': ['specific company email policies',
  'individual sender intentions',
  'future email content changes',
  'personal email preferences']

In [65]:
sample = dspy.Example(task="You have to judge the classification of the email given its body and subject",  context=email_context, output=ClassificationOutput(category="SPAM", confidence_score=0.95, rationale="As this email is not relevant to Crew Onboarding, it is classified as SPAM"))

sample_out_of_scope = dspy.Example(
    task="You have to judge the classification of the email given its body and subject",
    context="""Subject: Lunch Menu for Next Week

Hi Team,

Please find attached the lunch menu for next week at the cafeteria. Let me know if you have any dietary restrictions.

Best,
Admin""",
    output=ClassificationOutput(
        category="OUT_OF_SCOPE",
        confidence_score=0.98,
        rationale="The email content is about cafeteria lunch menu, which is unrelated to email delivery or policy issues."
    )
)

sample_spam = dspy.Example(
    task="You have to judge the classification of the email given its body and subject",
    context="""Subject: Congratulations! You've won a free cruise

Dear User,

You have been selected for a free cruise to the Bahamas. Click the link below to claim your prize.

Best regards,
Cruise Promotions""",
    output=ClassificationOutput(
        category="SPAM",
        confidence_score=0.99,
        rationale="The email contains promotional content and a suspicious link, which are typical indicators of spam."
    )
)

sample_policy_inquiry = dspy.Example(
    task="You have to judge the classification of the email given its body and subject",
    context="""Subject: Question about Email Retention Policy

Hello,

Could you please clarify how long our emails are stored on the company server? I want to ensure compliance with our data retention guidelines.

Thanks,
Employee""",
    output=ClassificationOutput(
        category="POLICY_INQUIRY",
        confidence_score=0.97,
        rationale="The sender is asking about company email retention policy, which is a direct policy inquiry."
    )
)


all_samples = [sample, sample_out_of_scope, sample_spam, sample_policy_inquiry]




In [70]:
def validate_category(example, prediction):
    return example.category == prediction.category

In [74]:
optimizer = dspy.teleprompt.MIPROv2

# opt_model = optimizer.compile(
# )

lm = dspy.LM(
    model='bedrock/us.anthropic.claude-3-5-sonnet-20241022-v2:0',
    max_tokens=4096
)

prompt_gen_lm = dspy.LM(
    model='bedrock/us.anthropic.claude-3-7-sonnet-20250219-v1:0',
    max_tokens=4096 
)

# Optimize
tp = dspy.MIPROv2(metric=validate_category, auto="light", prompt_model=prompt_gen_lm, task_model=lm)
optimized_classify = tp.compile(model, trainset=all_samples, max_labeled_demos=0, max_bootstrapped_demos=0, requires_permission_to_run=False)

2025/08/05 00:10:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 3

2025/08/05 00:10:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/05 00:10:50 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used for informing instruction proposal.

2025/08/05 00:10:50 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6


100%|██████████| 1/1 [00:00<00:00, 2874.78it/s]


Bootstrapped 0 full traces after 0 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 3/6


100%|██████████| 1/1 [00:00<00:00, 4021.38it/s]


...

2025/08/05 00:11:07 ERROR dspy.utils.parallelizer: Error for Example({'task': 'You have to judge the classification of the email given its body and subject', 'context': 'Subject: Lunch Menu for Next Week\n\nHi Team,\n\nPlease find attached the lunch menu for next week at the cafeteria. Let me know if you have any dietary restrictions.\n\nBest,\nAdmin', 'output': ClassificationOutput(category='OUT_OF_SCOPE', confidence_score=0.98, rationale='The email content is about cafeteria lunch menu, which is unrelated to email delivery or policy issues.')}) (input_keys=None): Inputs have not been set for this example. Use `example.with_inputs()` to set them.. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%): 100%|██████████| 3/3 [00:00<00:00, 2843.60it/s]

2025/08/05 00:11:07 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 0.0 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 3', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 4'].
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 0.0
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 21 / 20 =====
2025/08/05 00:11:07 ERROR dspy.utils.parallelizer: Error for Example({'task': 'You have to judge the classification of the email given its body and subject', 'context': "Subject: Congratulations! You've won a free cruise\n\nDear User,\n\nYou have been selected for a free


Average Metric: 0.00 / 0 (0%): 100%|██████████| 3/3 [00:00<00:00, 2821.91it/s]

2025/08/05 00:11:07 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 3 (0.0%)
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 0.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 3', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 1'].
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 0.0
2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/08/05 00:11:07 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 0.0!


In [86]:
print(optimized_classify.persona_predictor.history[0]['outputs'][0])

[[ ## reasoning ## ]]
For judging email classification, we need a persona with expertise in communication analysis, information organization, and data classification. The ideal persona should have:
1. Strong analytical skills to assess email content
2. Experience with business communication
3. Knowledge of email management and classification systems
4. Attention to detail and systematic thinking
5. Understanding of organizational workflows

[[ ## persona ## ]]
{
    "name": "Email Classification Specialist",
    "background": "Information Management Professional with 10+ years experience in email systems and content classification",
    "personality_traits": [
        "detail-oriented",
        "systematic",
        "methodical",
        "precise",
        "objective"
    ],
    "communication_style": "analytical",
    "domain_expertise": [
        "email classification systems",
        "content analysis",
        "information architecture",
        "business communication",
        "

In [92]:
print(optimized_classify.persona_agent.history[0]['messages'][0]['content'])

Your input fields are:
1. `role_persona` (RolePersona): Detailed description of the role to adopt
2. `task` (str): The task to complete in character
3. `context` (str): Additional context for the task
Your output fields are:
1. `reasoning` (str): 
2. `classification_output` (ClassificationOutput): Response maintaining the specified persona
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## role_persona ## ]]
{role_persona}

[[ ## task ## ]]
{task}

[[ ## context ## ]]
{context}

[[ ## reasoning ## ]]
{reasoning}

[[ ## classification_output ## ]]
{classification_output}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "description": "Output of the classification task", "properties": {"category": {"type": "string", "description": "The category of the classification", "enum": ["OUT_OF_SCOPE", "SPAM", "POLICY_INQUIRY"], "title": "Category"}, "confidence_score": {"type": "number", "description": "Th